# Swing-Up v2 — Single SAC Agent

Rebuild of the two-PPO hybrid. The v1 post-mortem showed 100% swing-up reach and 55% stabilizer hold rate but only **30% end-to-end success** — the seam between policies was the bottleneck.

| | v1 (two-PPO) | v2 (this notebook) |
|---|---|---|
| **Agents** | swing-up PPO + stabilizer PPO | single SAC agent |
| **Handoff** | angle threshold → distribution shift | none |
| **Architecture** | [64, 64] MLP | [256, 256] MLP |
| **Balance bonus** | none | +3 when both \|θ\| < 0.10 rad **and** \|θ̇\| < 0.50 rad/s |

**Why SAC?** Entropy maximisation actively explores state space, making the upright attractor discoverable from a hanging start without a curriculum. Off-policy replay is also more sample-efficient than PPO at the same wall-clock budget.

**Why a velocity gate on the bonus?** In v1 the energy controller passed through upright with \|θ̇\| ≈ 2 rad/s and was considered "near balanced" (angle-only gate). The velocity gate prevents the agent from farming the bonus by swinging through the top rapidly.

In [ ]:
import sys
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import gymnasium as gym
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import EvalCallback

CWD = pathlib.Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [CWD, *CWD.parents] if (p / "pyproject.toml").exists())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Config ──────────────────────────────────────────────────────────────────
ENV_ID         = "InvertedDoublePendulum-v5"
MODEL_DIR      = PROJECT_ROOT / "data" / "sac_swingup"
SAC_STEPS      = 2_000_000

ANGLE_THRESH   = 0.10   # rad  — balance gate (~6°)
VEL_THRESH     = 0.50   # rad/s — balance gate
BALANCE_BONUS  = 3.0    # added to height reward inside the gate
CART_LIMIT     = 2.4    # m    — termination
ANGLE_NOISE    = 0.05   # rad  — init noise around hanging
BALANCE_STEPS  = 200    # consecutive steps inside gate → success (10 s)
MAX_EP_STEPS   = 5_000  # evaluation cap
TRAIN_EP_STEPS = 1_000  # training cap — faster episode turnover
N_EVAL         = 20

MODEL_DIR.mkdir(parents=True, exist_ok=True)
_env    = gym.make(ENV_ID)
DT      = _env.unwrapped.dt
OBS_DIM = _env.observation_space.shape[0]
ACT_DIM = _env.action_space.shape[0]
_env.close()

print(f"DT={DT}s  OBS={OBS_DIM}  ACT={ACT_DIM}")
print(f"Balance gate : |θ| < {np.degrees(ANGLE_THRESH):.0f}°  |θ̇| < {VEL_THRESH} rad/s")
print(f"Success      : {BALANCE_STEPS} consecutive balanced steps ({BALANCE_STEPS*DT:.0f} s)")

In [ ]:
def obs_to_state6(obs: np.ndarray) -> np.ndarray:
    """9-dim MuJoCo obs → [x, θ₁, θ₂, ẋ, θ̇₁, θ̇₂]."""
    return np.array([
        obs[0],
        np.arctan2(obs[1], obs[3]),
        np.arctan2(obs[2], obs[4]),
        obs[5], obs[6], obs[7],
    ], dtype=np.float64)


def near_balanced(obs: np.ndarray) -> bool:
    """True when both poles are near vertical AND moving slowly."""
    th1 = np.arctan2(obs[1], obs[3])
    th2 = np.arctan2(obs[2], obs[4])
    return (
        abs(th1) < ANGLE_THRESH and abs(th2) < ANGLE_THRESH
        and abs(obs[6]) < VEL_THRESH and abs(obs[7]) < VEL_THRESH
    )

---

## Environment Wrapper

`SwingUpSACWrapper` makes two changes to `InvertedDoublePendulum-v5`:

| | Original | Wrapper |
|---|---|---|
| **Init** | near upright | hanging (θ₁ ≈ π) |
| **Reward** | alive bonus − distance penalty | height reward + balance gate bonus |
| **Termination** | tip height < 1 m | only if \|x\| > 2.4 m |

**Reward**:
```
r = cos θ₁ + cos(θ₁+θ₂) − 0.01 x²   ← height term, ranges [−2, +2]
  + BALANCE_BONUS                       ← if near_balanced(obs)
```

The height term gives a dense gradient everywhere — no reward cliff. The balance bonus is only collected when *both* angles are small *and* angular velocities are low, so the agent cannot farm it by swinging through the top quickly.

In [ ]:
class SwingUpSACWrapper(gym.Wrapper):
    def reset(self, seed=None, options=None):
        obs, info = self.env.reset(seed=seed, options=options)
        env  = self.unwrapped
        qpos = env.data.qpos.copy()
        qpos[0] = 0.0
        qpos[1] = np.pi + env.np_random.uniform(-ANGLE_NOISE, ANGLE_NOISE)
        qpos[2] = env.np_random.uniform(-ANGLE_NOISE, ANGLE_NOISE)
        env.set_state(qpos, np.zeros(env.model.nv))
        return env._get_obs(), info

    def step(self, action):
        obs, _, _, truncated, info = self.env.step(action)
        cos_th1     = obs[3]
        cos_th1_th2 = obs[3] * obs[4] - obs[1] * obs[2]
        height_rew  = cos_th1 + cos_th1_th2 - 0.01 * obs[0] ** 2
        reward      = height_rew + (BALANCE_BONUS if near_balanced(obs) else 0.0)
        terminated  = bool(abs(obs[0]) > CART_LIMIT)
        return obs, reward, terminated, truncated, info


# Sanity check: reward at hanging should be −2, gate should be False
_env_check = SwingUpSACWrapper(gym.make(ENV_ID))
obs_hang, _ = _env_check.reset(seed=0)
h = obs_hang[3] + obs_hang[3]*obs_hang[4] - obs_hang[1]*obs_hang[2]
print(f"Height reward at hanging : {h:.2f}  (min=−2, max=+2)")
print(f"near_balanced at hanging : {near_balanced(obs_hang)}")
_env_check.close()

---

## SAC Training

Single agent, architecture `[256, 256]`, all other hyperparameters are SB3 defaults.

- `ent_coef='auto'`: automatic entropy tuning — SAC adjusts the exploration bonus to match a target entropy, so we don't need to tune it manually.
- `TRAIN_EP_STEPS=1000` (50 s): shorter than the eval cap so failed episodes reset quickly and the replay buffer sees diverse starts.
- `EvalCallback` saves the best model by mean reward every 25k steps.

In [ ]:
BEST_MODEL  = MODEL_DIR / "best_model.zip"
FINAL_MODEL = MODEL_DIR / "sac_final.zip"

model_path = FINAL_MODEL if FINAL_MODEL.exists() else (BEST_MODEL if BEST_MODEL.exists() else None)

if model_path is not None:
    print(f"Loading SAC from {model_path}")
    model = SAC.load(str(model_path))
else:
    print(f"Training SAC for {SAC_STEPS:,} steps ...")
    train_env = SwingUpSACWrapper(gym.make(ENV_ID, max_episode_steps=TRAIN_EP_STEPS))
    eval_env  = SwingUpSACWrapper(gym.make(ENV_ID, max_episode_steps=MAX_EP_STEPS))

    model = SAC(
        "MlpPolicy", train_env,
        learning_rate=3e-4,
        buffer_size=1_000_000,
        batch_size=256,
        tau=0.005,
        gamma=0.99,
        ent_coef="auto",
        policy_kwargs=dict(net_arch=[256, 256]),
        verbose=0,
        tensorboard_log=str(MODEL_DIR / "tb"),
    )
    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path=str(MODEL_DIR),
        log_path=str(MODEL_DIR),
        eval_freq=25_000,
        n_eval_episodes=10,
        deterministic=True,
        verbose=1,
    )
    model.learn(total_timesteps=SAC_STEPS, callback=eval_cb, progress_bar=True)
    model.save(str(MODEL_DIR / "sac_final"))
    train_env.close(); eval_env.close()
    print("Saved.")

---

## Evaluation

Each episode starts from hanging and runs until:
- **Success**: both \|θ\| < 0.10 rad and \|θ̇\| < 0.50 rad/s for 200 consecutive steps (10 s)
- **Failure**: cart hits the boundary (\|x\| > 2.4 m) or 250 s elapsed

In [ ]:
def run_episode(model, seed: int = 0):
    """Roll out from hanging. Returns (states, actions, rewards, success)."""
    env = SwingUpSACWrapper(gym.make(ENV_ID, max_episode_steps=MAX_EP_STEPS))
    obs, _ = env.reset(seed=seed)

    states, actions, rewards = [], [], []
    consecutive_balanced = 0
    success = False

    for _ in range(MAX_EP_STEPS):
        action, _ = model.predict(obs, deterministic=True)
        states.append(obs_to_state6(obs))
        actions.append(float(action[0]))
        obs, r, terminated, truncated, _ = env.step(action)
        rewards.append(r)

        if near_balanced(obs):
            consecutive_balanced += 1
            if consecutive_balanced >= BALANCE_STEPS:
                success = True
                break
        else:
            consecutive_balanced = 0

        if terminated or truncated:
            break

    env.close()
    return np.array(states), np.array(actions), np.array(rewards), success

In [ ]:
print(f"Evaluating over {N_EVAL} episodes ...")
ep_lens, successes = [], []
for seed in range(N_EVAL):
    s, a, r, success = run_episode(model, seed=seed)
    ep_lens.append(len(s))
    successes.append(success)

n_success = sum(successes)
ok_lens   = [ep_lens[i] * DT for i in range(N_EVAL) if successes[i]]
print(f"  Success            : {n_success}/{N_EVAL}  ({100*n_success/N_EVAL:.0f}%)")
if ok_lens:
    print(f"  Mean time (success): {np.mean(ok_lens):.1f}s  "
          f"(range {min(ok_lens):.1f}–{max(ok_lens):.1f}s)")
print(f"  Mean episode time  : {np.mean(ep_lens)*DT:.1f}s")

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(
    np.arange(N_EVAL),
    [l * DT for l in ep_lens],
    color=["seagreen" if s else "steelblue" for s in successes],
    alpha=0.75, width=0.8,
)
ax.set_xlabel("Episode seed")
ax.set_ylabel("Time (s)")
ax.set_title(f"SAC swing-up — {n_success}/{N_EVAL} success")
ax.legend(handles=[
    mpatches.Patch(color="seagreen",  alpha=0.7, label="SUCCESS"),
    mpatches.Patch(color="steelblue", alpha=0.7, label="FAILED"),
], fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
def plot_episode(states, actions, rewards, success, title="SAC swing-up"):
    t      = np.arange(len(states)) * DT
    status = "SUCCESS" if success else "FAILED"
    color  = "seagreen" if success else "crimson"

    fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)

    axes[0].plot(t, np.degrees(states[:, 1]), label="θ₁", color="steelblue")
    axes[0].plot(t, np.degrees(states[:, 2]), label="θ₂", color="darkorange")
    axes[0].axhline(0, ls="--", color="gray", lw=0.8)
    axes[0].axhline( np.degrees(ANGLE_THRESH), ls=":", color="seagreen", lw=0.8)
    axes[0].axhline(-np.degrees(ANGLE_THRESH), ls=":", color="seagreen", lw=0.8)
    axes[0].set_ylabel("Angle (deg)")
    axes[0].legend(fontsize=9)

    axes[1].plot(t, states[:, 0], color="steelblue")
    axes[1].axhline(0, ls="--", color="gray", lw=0.8)
    axes[1].set_ylabel("Cart x (m)")

    axes[2].plot(t, actions, color="crimson", lw=0.8)
    axes[2].axhline(0, ls="--", color="gray", lw=0.8)
    axes[2].set_ylabel("Action")

    axes[3].plot(t, rewards, color="goldenrod", lw=0.8)
    axes[3].axhline(2.0, ls=":", color="seagreen", lw=0.8, label="max height reward")
    axes[3].set_ylabel("Reward")
    axes[3].set_xlabel("Time (s)")
    axes[3].legend(fontsize=9)

    fig.suptitle(f"{title}  |  {len(states)*DT:.1f}s  [{status}]",
                 fontsize=11, color=color)
    plt.tight_layout()
    plt.show()


# Show first successful episode, or seed 0 if none succeeded
best_seed = next((i for i in range(N_EVAL) if successes[i]), 0)
states, actions, rewards, success = run_episode(model, seed=best_seed)
print(f"Seed {best_seed}: {len(states)} steps ({len(states)*DT:.1f}s)  "
      f"[{'SUCCESS' if success else 'FAILED'}]")
plot_episode(states, actions, rewards, success)

---

## Animation

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML


def render_episode(model, seed: int = 0, speed: int = 3) -> HTML:
    env = SwingUpSACWrapper(gym.make(ENV_ID, render_mode="rgb_array",
                                     max_episode_steps=MAX_EP_STEPS))
    obs, _ = env.reset(seed=seed)
    frames = [env.render()]
    consecutive_balanced = 0
    success = False
    balance_marker = None

    for step in range(MAX_EP_STEPS):
        action, _ = model.predict(obs, deterministic=True)
        obs, _, terminated, truncated, _ = env.step(action)
        frames.append(env.render())

        if near_balanced(obs):
            consecutive_balanced += 1
            if consecutive_balanced >= BALANCE_STEPS:
                success = True
                if balance_marker is None:
                    balance_marker = step - BALANCE_STEPS + 1
                break
        else:
            consecutive_balanced = 0

        if terminated or truncated:
            break
    env.close()

    n_steps = len(frames) - 1
    status  = "SUCCESS" if success else "FAILED"
    print(f"{n_steps} steps ({n_steps*DT:.1f}s)  [{status}]")

    fig, ax = plt.subplots(figsize=(4, 6))
    ax.axis("off")
    im = ax.imshow(frames[0])

    def _update(i):
        t = i * speed
        is_balanced = (balance_marker is not None and t >= balance_marker)
        fig.patch.set_facecolor("honeydew" if is_balanced else "aliceblue")
        im.set_data(frames[min(t, len(frames) - 1)])
        ax.set_title(f"t={t*DT:.1f}s", fontsize=9,
                     color="seagreen" if is_balanced else "royalblue")
        return [im]

    ani = animation.FuncAnimation(
        fig, _update,
        frames=range(len(frames) // speed + 1),
        interval=50, blit=False,
    )
    plt.close(fig)
    return HTML(ani.to_jshtml())


render_episode(model, seed=best_seed, speed=3)

---

## Bonus Magnitude Sensitivity

The model was trained with `BALANCE_BONUS=3.0`. Here we ask: how sensitive is the success rate to the exact value of the threshold used at *evaluation* time — i.e., does the agent actually achieve tight balance or just barely scrape the gate?

For each threshold pair `(angle_thresh, vel_thresh)`, we re-run the same 20 episodes with a stricter or looser success criterion (but the *policy is unchanged*).

In [ ]:
ANGLE_THRESHOLDS = [0.05, 0.10, 0.15, 0.20]
VEL_THRESHOLDS   = [0.25, 0.50, 1.00]

results = np.zeros((len(ANGLE_THRESHOLDS), len(VEL_THRESHOLDS)))

# Pre-collect trajectories once (policy unchanged)
all_trajectories = [run_episode(model, seed=s) for s in range(N_EVAL)]

for ai, at in enumerate(ANGLE_THRESHOLDS):
    for vi, vt in enumerate(VEL_THRESHOLDS):
        n_ok = 0
        for states, actions, rewards, _ in all_trajectories:
            consec = 0
            for state in states:
                th1, th2 = state[1], state[2]
                th1d, th2d = state[4], state[5]
                if abs(th1) < at and abs(th2) < at and abs(th1d) < vt and abs(th2d) < vt:
                    consec += 1
                    if consec >= BALANCE_STEPS:
                        n_ok += 1
                        break
                else:
                    consec = 0
        results[ai, vi] = 100 * n_ok / N_EVAL

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(results, aspect="auto", vmin=0, vmax=100, cmap="YlGn")
ax.set_xticks(range(len(VEL_THRESHOLDS)))
ax.set_yticks(range(len(ANGLE_THRESHOLDS)))
ax.set_xticklabels([f"{v} rad/s" for v in VEL_THRESHOLDS])
ax.set_yticklabels([f"{np.degrees(a):.0f}°" for a in ANGLE_THRESHOLDS])
ax.set_xlabel("vel_thresh")
ax.set_ylabel("angle_thresh")
ax.set_title("Success % at different evaluation thresholds (same trained policy)")
for ai in range(len(ANGLE_THRESHOLDS)):
    for vi in range(len(VEL_THRESHOLDS)):
        ax.text(vi, ai, f"{results[ai, vi]:.0f}%", ha="center", va="center", fontsize=11)
plt.colorbar(im, ax=ax, label="Success %")
plt.tight_layout()
plt.show()